# P2 · House prices: from the prediction to trust in the prediction

**Type**: Implement with scikit-learn · **Level**: ●●● challenge · **Time**: about 100 minutes · **core path ★**

## Objective

Build a regression model with XGBoost on California housing and then answer two questions a customer would really ask: **how reliable** is the number you estimate if the houses to predict are in an area never seen (spatial validation) and **with what margin** (90% prediction intervals with quantile regression and their coverage).

## Dataset

`ds.california()`: 20640 districts, 8 features (`MedInc`, `HouseAge`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`, `Latitude`, `Longitude`); target: median value (in hundreds of thousands of dollars, capped at 5). 80/20 split (`X_tr, X_te, y_tr, y_te`, given). `new_xgb()`: `XGBRegressor(400 trees, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8)`.

## Educational purpose

Learn that the estimated error depends on the **validation protocol**: with a random partition neighboring districts are in both the training and the validation set, and the error comes out much lower than the one you will have with new areas. And how to give an honest margin of uncertainty, checking that the interval really covers 90% of the cases (and not only where it was calibrated).

## Task

1. **Feature engineering:** `enrich(X)` (returns a copy with `rooms_per_person = AveRooms / AveOccup`, `bedroom_share = AveBedrms / AveRooms`, `d_LA`, `d_SF`, `d_SD` (Euclidean distance `np.hypot(Latitude − lat, (Longitude − lon) * 0.8)` from Los Angeles, San Francisco and San Diego: coordinates in `CITIES`) and `d_city`, the minimum of the three). Save `base_rmse` (XGBoost on `X_tr`, RMSE on the test set) and `fe_rmse` (with `enrich`).
2. **Random and spatial validation:** `blocks` (for each row of `X_tr` the string `"{floor(Latitude)}_{floor(Longitude)}"`: cells of 1° by 1°) and `n_blocks`; with `enrich(X_tr)` and `new_xgb()` compute `random_rmse` (`KFold(5, shuffle=True, random_state=0)`) and `spatial_fold_rmse` (list of the 5 RMSEs of `GroupKFold(5)` with `groups=blocks`, scoring `neg_root_mean_squared_error`), `spatial_rmse` (the mean) and `overestimate = spatial_rmse / random_rmse − 1`.
3. **Prediction intervals:** on a subsample of 10000 training rows (`idx = np.random.default_rng(0).permutation(len(X_tr))[:10000]`, `enrich` features) train two `GradientBoostingRegressor(loss="quantile", alpha=..., n_estimators=150, max_depth=4, learning_rate=0.1, subsample=0.8, random_state=0)` with `alpha = 0.05` and `0.95`. Save `test_coverage` (fraction of `y_te` in the interval `[q05, q95]`) and `mean_width`.
4. **Spatial coverage:** for each of the 5 folds of `GroupKFold(5)` (groups `blocks`) train the same two models on the training part of the fold (subsample of 10000 rows with `np.random.default_rng(0).permutation(len(a))[:10000]`) and measure the coverage on the validation part: `spatial_coverages` (list), `mean_spatial_coverage` and `min_coverage`.
5. **Report:** `report` with `test_rmse` (=`fe_rmse`), `random_rmse`, `spatial_rmse`, `test_coverage`, `mean_spatial_coverage` and `message` (a sentence for the customer: how much you can expect to be wrong in a new area and with what margin).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold, KFold, cross_val_score, train_test_split

X, y = ds.california()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)
CITIES = {"LA": (34.05, -118.24), "SF": (37.77, -122.42), "SD": (32.72, -117.16)}


def new_xgb():
    return xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=0)


def quantile_model(alpha):
    return GradientBoostingRegressor(loss="quantile", alpha=alpha, n_estimators=150, max_depth=4, learning_rate=0.1, subsample=0.8, random_state=0)

## Your work

### 1 · Feature engineering

In [ ]:
def enrich(X):
    ...


base_rmse = fe_rmse = None

### 2 · Random and spatial validation

In [ ]:
blocks = None
n_blocks = None
random_rmse = None
spatial_fold_rmse = []
spatial_rmse = None
overestimate = None

### 3-4 · Intervals and coverage

In [ ]:
test_coverage = None
mean_width = None
spatial_coverages = []
mean_spatial_coverage = None
min_coverage = None

### 5 · Report

In [ ]:
report = {}

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`blocks = (np.floor(X_tr["Latitude"]).astype(int).astype(str) + "_" + np.floor(X_tr["Longitude"]).astype(int).astype(str)).to_numpy()`. CV: `cross_val_score(new_xgb(), Fe, y_tr, cv=GroupKFold(5), groups=blocks, scoring="neg_root_mean_squared_error")` and then `-s`. Coverage: `((y_val >= q05.predict(Xv)) & (y_val <= q95.predict(Xv))).mean()`.'):
    Fe = enrich(X_tr)
    check({"rooms_per_person", "bedroom_share", "d_LA", "d_SF", "d_SD", "d_city"} <= set(Fe.columns) and len(Fe) == len(X_tr) and np.isfinite(Fe.to_numpy()).all() and (Fe["d_city"] <= Fe[["d_LA", "d_SF", "d_SD"]].max(axis=1)).all(), "six new columns, with no NaN or infinities, d_city is the minimum", "np.hypot(Latitude - lat, (Longitude - lon) * 0.8)")
    check(base_rmse is not None and fe_rmse < base_rmse - 0.008 and fe_rmse < 0.44, "the new features reduce the test RMSE by over 0.008 (below 0.44)", "XGBoost on enrich(X_tr), test on enrich(X_te)")
    check(blocks is not None and len(blocks) == len(X_tr) and 40 <= n_blocks <= 70, "between 40 and 70 blocks of 1 degree by 1 degree", "np.floor(Latitude), np.floor(Longitude)")
    check(0.42 < random_rmse < 0.48 and len(spatial_fold_rmse) == 5 and close(spatial_rmse, np.mean(spatial_fold_rmse)), "RMSE with random CV between 0.42 and 0.48; five spatial folds", "scoring='neg_root_mean_squared_error' (flip the sign)")
    check(overestimate > 0.25 and max(spatial_fold_rmse) > 1.4 * min(spatial_fold_rmse), "the spatial error exceeds the random one by over 25% and the spatial folds differ by over 40% among themselves", "spatial_rmse / random_rmse - 1")
    check(test_coverage is not None and 0.80 < test_coverage < 0.93 and mean_width > 1.0, "coverage on the test set between 0.80 and 0.93 (nominal 0.90); mean width over 1.0 (100 thousand dollars)", "(y_te >= q05) & (y_te <= q95)")
    check(len(spatial_coverages) == 5 and mean_spatial_coverage < test_coverage - 0.03 and min_coverage < 0.8, "in new areas the mean coverage drops by over 3 points below that of the test set and in the worst fold it is below 0.80", "GroupKFold(5) with the blocks")
    check(set(report) >= {"test_rmse", "random_rmse", "spatial_rmse", "test_coverage", "mean_spatial_coverage", "message"} and isinstance(report["message"], str) and len(report["message"]) > 40, "complete report", "a dictionary with the required keys")
done()

---
The commented solution is in `solutions/B_with_sklearn/P2_house_prices_from_prediction_to_trust_in_the_prediction.ipynb`. Open it after you pass the check, or after you have really tried.